# DeBCR API tutorial
## Deblur data using trained DeBCR model

This notebook shows how to restore low-quality microscopy data using DeBCR.

To achieve that you would need:
- **pre-processed input data** by normalization and patching;
- **trained DeBCR model** on the same-sized data as the input.

Please find on the DeBCR GitHub page links to:
- notebook tutorial on raw data pre-processing protocol; 
- samples, i.e. examples of pre-processed input data and trained DeBCR model weights.

In [ ]:
import debcr

### Load pre-processed input data

Set file path to your actual pre-processed input data (in NPZ or NPY format).

For sample data: ```/path/to/examples/DATASET/data/DATASET_test.npz```

In [ ]:
data_filepath = '/path/to/data.npz'
data_filepath

Load input data

In [ ]:
data = debcr.data.load(data_filepath)

The example input data is provided as multi-array NPZ file, which contains two arrays: 
- "low" - input data (low-quality data to be improved)
- "gt" - ground-truth data for comparison

You can check the filenames as below

In [ ]:
data.files

and the respective array size:

In [ ]:
data["low"].shape, data["gt"].shape

### Visualize loaded input data

for sample data

In [ ]:
debcr.data.show(
    data = [data["low"], data["gt"]],
    slices = [250, 500, -1], # -1 is to pick a random slice
    titles = ['input', 'ground truth'],
    transpose=True
)

or for custom data

In [ ]:
debcr.data.show(
    data = [data],
    slices = [-1, -1, -1], # -1 is to pick a random slice
    titles = ['input'],
    transpose=True
)

### Load trained DeBCR model

Set directory path to your actual trained DeBCR model weights.

For sample data: ```/path/to/examples/DATASET/weights/```

In [ ]:
weights_dirpath = '/path/to/weights'
weights_dirpath

Load trained DeBCR model

In [ ]:
debcr_model = debcr.model.init(weights_dirpath)

The model you intend to use for prediction should had been trained on the same-sized data, as the intended prediction input.

Thus, to correctly load the model you need to provide `input_size` value (unless default size was used for training, like for samples):

```debcr_model = debcr.model.init(weights_dirpath, input_size=128)```

Show TensorFlow model info to see model structure details (e.g. to verify input size) 

In [ ]:
debcr_model.summary()

### Run DeBCR prediction

In [ ]:
# (un)comment respective line

# data_in = data['low'] # for samples data
data_in = data # for custom data

To run prediction simply pass the loaded trained DeBCR model and input data.

By changing the `batch_size` you can manage amount of data to be processed at once for GPU memory control purposes.

In [ ]:
data_pred = debcr.model.predict(eval_model=debcr_model, input_data=data_in, batch_size=32)
data_pred.shape

### Visualize predictions

for sample data

In [ ]:
debcr.data.show(
    data = [data_in, data_pred, data["gt"]],
    slices = [250, 500, -1],
    titles = ['input', 'prediction', 'ground truth']
)

or for custom data

In [ ]:
debcr.data.show(
    data = [data_in, data_pred],
    slices = [-1, -1, -1],
    titles = ['input', 'prediction'],
    transpose=True
)

### Stitch prediction (optional: when cropping params are known)

First of all, the stitching is only possible if cropping parameters are known.

Here we show the stitching example using parameters for the provided 'FM_CARE_2D' sample dataset (the dataset link is on DeBCR GitHub).

The overlap was known, while the patches amount were obtained by the dry-run of `debcr.data.crop` (see data pre-processing tutorial).

In [ ]:
data_asmbl = debcr.data.stitch(data_pred, patch_num=(15,15), overlap=(0.5, 0.5))
data_asmbl.shape

Let's also visualize result of the stitching

In [ ]:
debcr.data.show([data_asmbl], slices=[50,70,90], cmap='gray', transpose=True)

### Save prediction

Finally, let's save predictions, either patched (for sample data) or already assembled.

In [ ]:
data_pred_filepath = '/path/to/save/pred.npy'
data_pred_filepath

In [ ]:
# (un)comment respective line

# data_ou = data_pred # for predicted sample patches
data_ou = data_asmbl # for predicted and assembled data

In [ ]:
debcr.data.write(data_pred_filepath, data=data_ou)

If you only used the provided sample data so far, try to adapt the whole pipeline (preprocess-train-predict) for your own data.

Otherwise, good luck with deblurring using DeBCR!